# Trittention: Visualizing N-way Attention

This notebook demonstrates the different attention mechanisms implemented in the Trittention-Transformer project, with a focus on visualizing attention patterns and understanding the differences between standard attention and trittention variants.

## Setup and Imports

First, let's import the necessary modules and set up our environment.

In [ ]:
# For Google Colab, uncomment the following to install the package from GitHub
# !pip install git+https://github.com/muditbhargava66/Trittention-Transformer.git
# !git clone https://github.com/muditbhargava66/Trittention-Transformer.git
# %cd Trittention-Transformer

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path if running notebook in examples directory
if os.path.basename(os.getcwd()) == 'examples':
    sys.path.append('..')

from config.cfgs import TrittentionConfig
from models import (
    Attention,
    Trittention,
    TrittentionCube,
    LocalTrittention,
    SparseTrittention,
    WindowedTrittention
)
from utils.visualization_utils import (
    visualize_attention_matrix,
    visualize_attention_comparisons,
    plot_complexity_analysis
)

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Creating Attention Models

Let's initialize different attention mechanisms with a common configuration.

In [ ]:
# Create a configuration
config = TrittentionConfig(
    hidden_size=64,
    num_attention_heads=4,
    attention_probs_dropout_prob=0.0,  # Disable dropout for deterministic results
    window_size=16,  # For windowed/local attention
    sparsity_threshold=0.1  # For sparse trittention
)

# Initialize models
standard_attention = Attention(config)
trittention = Trittention(config)
trittention_cube = TrittentionCube(config)
sparse_trittention = SparseTrittention(config)
windowed_trittention = WindowedTrittention(config)

# Set all models to evaluation mode
models = {
    "Standard Attention": standard_attention,
    "Trittention": trittention,
    "Trittention Cube": trittention_cube,
    "Sparse Trittention": sparse_trittention,
    "Windowed Trittention": windowed_trittention
}

for model in models.values():
    model.eval()

## Generating Sample Data

We'll create a simple synthetic dataset to visualize attention patterns.

In [ ]:
# Generate sample input
batch_size = 1
seq_length = 20
hidden_size = config.hidden_size

# Create a structured input to get interesting attention patterns
# We'll create a pattern where every 4th token is related
hidden_states = torch.zeros(batch_size, seq_length, hidden_size)

# Create groups of related tokens
for i in range(4):  # 4 groups
    group_pattern = torch.randn(1, 1, hidden_size)  # Base pattern for this group
    for j in range(i, seq_length, 4):  # Every 4th token, starting from i
        # Add the group pattern plus some noise
        hidden_states[:, j, :] = group_pattern + 0.3 * torch.randn(1, hidden_size)

print(f"Input shape: {hidden_states.shape}")

## Computing Attention Patterns

Now, let's run our models and extract their attention patterns for visualization.

In [ ]:
# Function to extract attention patterns from different models
def get_attention_patterns(models, hidden_states):
    attention_patterns = {}
    
    for name, model in models.items():
        with torch.no_grad():
            # For models that support returning attention weights
            if hasattr(model, 'get_attention_weights') or hasattr(model, 'attention_weights'):
                # Forward pass with attention output
                if hasattr(model, 'forward') and 'output_attentions' in model.forward.__code__.co_varnames:
                    _, attn_weights = model(hidden_states, output_attentions=True)
                else:
                    _ = model(hidden_states)
                    # Try to get attention weights
                    if hasattr(model, 'get_attention_weights'):
                        attn_weights = model.get_attention_weights()
                    else:
                        attn_weights = model.attention_weights
                
                # For multi-head attention, take the average across heads
                if attn_weights.dim() > 3:  # [batch, heads, seq_len, seq_len]
                    attn_weights = attn_weights.mean(dim=1)
                
                # Extract the first batch item
                attention_patterns[name] = attn_weights[0].cpu()
            else:
                print(f"Warning: Could not extract attention patterns for {name}")
    
    return attention_patterns

In [ ]:
# Get attention patterns
attention_patterns = get_attention_patterns(models, hidden_states)

## Visualizing Attention Patterns

Let's visualize the attention patterns from different models.

In [ ]:
# Visualize standard attention
if "Standard Attention" in attention_patterns:
    visualize_attention_matrix(
        attention_patterns["Standard Attention"],
        title="Standard Attention Pattern",
        figsize=(10, 8)
    )

In [ ]:
# Visualize trittention
if "Trittention" in attention_patterns:
    visualize_attention_matrix(
        attention_patterns["Trittention"],
        title="Trittention Pattern",
        figsize=(10, 8)
    )

In [ ]:
# Compare all attention patterns
visualize_attention_comparisons(
    attention_patterns,
    title="Comparison of Attention Mechanisms",
    figsize=(20, 10)
)

## Analyzing Differences in Attention Patterns

Let's analyze the differences between standard attention and trittention patterns.

In [ ]:
# Calculate and visualize differences
if "Standard Attention" in attention_patterns and "Trittention" in attention_patterns:
    # Calculate difference
    difference = attention_patterns["Trittention"] - attention_patterns["Standard Attention"]
    
    # Visualize difference
    visualize_attention_matrix(
        difference,
        title="Difference: Trittention - Standard Attention",
        cmap="coolwarm",  # Use diverging colormap for differences
        figsize=(10, 8)
    )

## Performance Comparison

Now, let's benchmark the computational efficiency of different attention mechanisms across various sequence lengths.

In [ ]:
def benchmark_models(models, sequence_lengths, hidden_size=64, num_runs=3):
    # Dictionaries to store results
    time_results = {name: [] for name in models.keys()}
    
    import time
    
    # Run benchmarks for each sequence length
    for seq_len in sequence_lengths:
        print(f"Benchmarking sequence length: {seq_len}")
        
        # Create input tensor
        x = torch.randn(1, seq_len, hidden_size)
        
        # Benchmark each model
        for name, model in models.items():
            # Run multiple times and take the minimum
            run_times = []
            
            for _ in range(num_runs):
                # Warm-up run
                with torch.no_grad():
                    _ = model(x)
                
                # Timed run
                start_time = time.time()
                with torch.no_grad():
                    _ = model(x)
                run_times.append(time.time() - start_time)
            
            # Record the minimum time
            time_results[name].append(min(run_times))
    
    return time_results

In [ ]:
# Define sequence lengths to benchmark
sequence_lengths = [10, 20, 50, 100, 200, 500]

# Run benchmarks
time_results = benchmark_models(models, sequence_lengths)

In [ ]:
# Plot results
plot_complexity_analysis(
    sequence_lengths=sequence_lengths,
    time_complexities=time_results,
    title="Attention Mechanisms Complexity Analysis",
    figsize=(12, 6)
)

## Visualizing Attention for Text Data

Let's create a simple text example to better understand how different attention mechanisms work.

In [ ]:
from utils.visualization_utils import create_attention_map_for_text

# Define a simple sentence
text = ["The", "cat", "sat", "on", "the", "mat", ".", "It", "was", "happy", "."]

In [ ]:
# Create a simple tokenizer
def tokenize(tokens, vocab=None):
    if vocab is None:
        # Create vocabulary from tokens
        vocab = {token: i for i, token in enumerate(set(tokens))}
    
    # Convert tokens to indices
    indices = [vocab.get(token, vocab.get("<UNK>", 0)) for token in tokens]
    
    return torch.tensor(indices), vocab

In [ ]:
# Tokenize the text
token_ids, vocab = tokenize(text, vocab=None)
print(f"Vocabulary: {vocab}")
print(f"Token IDs: {token_ids}")

In [ ]:
# Create embedding layer
seq_length = len(text)
embedding_dim = config.hidden_size
embedding = torch.nn.Embedding(len(vocab), embedding_dim)

# Get embeddings
embeddings = embedding(token_ids).unsqueeze(0)  # Add batch dimension
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Get attention patterns for the text
text_attention_patterns = get_attention_patterns(models, embeddings)

In [ ]:
# Visualize attention patterns for the text
for name, pattern in text_attention_patterns.items():
    create_attention_map_for_text(
        text=text,
        attention_matrix=pattern,
        title=f"{name} for Text",
        figsize=(10, 8)
    )

## Summary and Conclusions

In this notebook, we've explored and visualized different attention mechanisms from the Trittention-Transformer project. Here are some key observations:

1. **Standard Attention** captures pairwise token relationships, showing a more localized pattern.
2. **Trittention** captures higher-order dependencies, potentially identifying more complex patterns in the data.
3. **Sparse Trittention** provides an efficient approximation of trittention by pruning less significant attention weights.
4. **Windowed Trittention** limits attention to a local window, significantly reducing computational complexity while maintaining performance.

The computational complexity analysis demonstrates the trade-offs between expressiveness and efficiency across different attention mechanisms. While standard attention scales as O(n²) and full trittention as O(n³), our optimized implementations (sparse and windowed) offer better scaling properties while preserving most of the modeling benefits.

These visualizations help us understand how different attention mechanisms process information and how they might be suitable for different types of tasks and sequence lengths.